# Projet : Rossmann Store Sales
Projet de prévision des ventes (variable cible : Sales), réparti en 5 tâches, du nettoyage à l’évaluation finale.

### Tâche 1 : initialisation du dépôt

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)

### Tâche 2 : structure du dataset

In [10]:
df = pd.read_csv("data/rossmann_stores_sales.csv")
sales_brut = df["Sales"].copy()   # gardée pour la comparaison avant/après

print("Taille :", df.shape)
print("\nTypes :\n", df.dtypes)
print("\nManquants :\n", df.isna().sum())
print("\nDoublons exacts :", df.duplicated().sum())
print("Doublons Store+Date :", df.duplicated(["Store", "Date"]).sum())
print("Période :", df["Date"].min(), "->", df["Date"].max())
print("Magasins :", df["Store"].nunique())

for c in ["StoreType", "Assortment", "StateHoliday", "PromoInterval"]:
    print(f"\n{c} :", df[c].map(repr).value_counts(dropna=False).to_dict())

print("\nSales négatives :", (df["Sales"] < 0).sum())
df["Sales"].describe()

Taille : (810056, 16)

Types :
 Store                          int64
Date                             str
DayOfWeek                      int64
Sales                        float64
Promo                          int64
StateHoliday                     str
SchoolHoliday                  int64
StoreType                        str
Assortment                       str
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                    str
dtype: object

Manquants :
 Store                             0
Date                              0
DayOfWeek                         0
Sales                           814
Promo                             0
StateHoliday                   8099
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance     

count    809242.000000
mean       6157.516553
std        3141.648269
min      -18554.000000
25%        4501.000000
50%        6321.000000
75%        8137.000000
max       21387.000000
Name: Sales, dtype: float64

### Tâche 3 : doublons, types, écriture des catégories

In [11]:
# Doublons
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Doublons supprimés : {n0 - len(df)} ({n0} -> {len(df)})")

# Date
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")

# Écriture des catégories
for c in ["StoreType", "Assortment"]:
    propre = df[c].str.strip().str.lower()
    print(f"{c} : {(df[c] != propre).sum()} lignes modifiées")
    df[c] = propre.astype("category")
    print("  modalités :", list(df[c].cat.categories))

Doublons supprimés : 6000 (810056 -> 804056)
StoreType : 63696 lignes modifiées
  modalités : ['a', 'b', 'c', 'd']
Assortment : 64591 lignes modifiées
  modalités : ['a', 'b', 'c']


### Tâche 4 : cohérence des valeurs et outliers

In [13]:
# Date / DayOfWeek (lundi = 1 ... dimanche = 7)
print("DayOfWeek incohérents :", ((df["Date"].dt.day_of_week + 1) != df["DayOfWeek"]).sum())

# Sales négatives : preuve de l'erreur de signe, puis correction
neg = df["Sales"] < 0
pos = df.loc[df["Sales"] > 0, "Sales"]
print("\nSales négatives (après doublons) :", neg.sum())
print("|négatives| :", df.loc[neg, "Sales"].abs().describe()[["min", "50%", "mean"]].to_dict())
print("positives   :", pos.describe()[["min", "50%", "mean"]].to_dict())
df.loc[neg, "Sales"] = df.loc[neg, "Sales"].abs()

# Ventes à 0
zeros = df[df["Sales"] == 0]
print("\nSales = 0 :", len(zeros), "| jours de la semaine :", zeros["DayOfWeek"].value_counts().to_dict())

# Outliers Sales (IQR sur Sales > 0) : conservés, signalés
pos = df.loc[df["Sales"] > 0, "Sales"]
q1, q3 = pos.quantile([0.25, 0.75])
borne = q3 + 1.5 * (q3 - q1)
print(f"Outliers Sales > {borne:.0f} :", (pos > borne).sum(), "| max :", df["Sales"].max())

# Outliers CompetitionDistance (niveau magasin)
cd = df.groupby("Store")["CompetitionDistance"].first().dropna()
a, b = cd.quantile([0.25, 0.75])
out = ((cd < a - 1.5 * (b - a)) | (cd > b + 1.5 * (b - a))).sum()
print("\nCompetitionDistance : min", round(cd.min(), 2), "max", round(cd.max(), 2), "| outliers IQR :", out)

# StateHoliday : dates portant a / b / c
sh = df.dropna(subset=["StateHoliday"])
for v in ["a", "b", "c"]:
    print(f"StateHoliday = {v} : {sh.loc[sh['StateHoliday'] == v, 'Date'].nunique()} dates distinctes")

DayOfWeek incohérents : 0

Sales négatives (après doublons) : 0
|négatives| : {'min': nan, '50%': nan, 'mean': nan}
positives   : {'min': 2574.0, '50%': 6689.0, 'mean': 6893.640290874088}

Sales = 0 : 80190 | jours de la semaine : {7: 80190}
Outliers Sales > 13259 : 6358 | max : 21387.0

CompetitionDistance : min 27.6 max 74973.32 | outliers IQR : 0
StateHoliday = a : 718 dates distinctes
StateHoliday = b : 718 dates distinctes
StateHoliday = c : 1 dates distinctes


### Tâche 5 : valeurs manquantes

In [14]:
# 1. Sales : cible manquante -> suppression
n = df["Sales"].isna().sum()
df = df.dropna(subset=["Sales"]).reset_index(drop=True)
df["Sales"] = df["Sales"].astype("int64")
print(f"Sales manquantes supprimées : {n} -> {len(df)} lignes")

# 2. StateHoliday : valeur majoritaire du même jour
df["StateHoliday"] = df["StateHoliday"].astype("string")
mode_jour = df.groupby("Date")["StateHoliday"].agg(lambda s: s.dropna().mode().iloc[0])
manq = df["StateHoliday"].isna()
print(f"StateHoliday manquants : {manq.sum()} dont 'c' : {(manq & (df['Date'].map(mode_jour) == 'c')).sum()}")
df["StateHoliday"] = df["StateHoliday"].fillna(df["Date"].map(mode_jour))

# 3. CompetitionDistance : valeur du même magasin, sinon médiane des magasins
avant = df["CompetitionDistance"].isna().sum()
df["CompetitionDistance"] = df.groupby("Store")["CompetitionDistance"].transform(lambda s: s.ffill().bfill())
reste = df["CompetitionDistance"].isna().sum()
mediane = df.groupby("Store")["CompetitionDistance"].first().median()
print(f"CompetitionDistance : {avant} manquants, {avant - reste} récupérés via le magasin, "
      f"{reste} remplis par la médiane ({mediane:.2f}) pour {df.loc[df['CompetitionDistance'].isna(), 'Store'].nunique()} magasins")
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(mediane)

# 4. CompetitionOpenSince* : 0 = inconnu / sans objet
print("CompetitionOpenSince* manquants :", df["CompetitionOpenSinceMonth"].isna().sum(),
      "| magasins :", df.loc[df["CompetitionOpenSinceMonth"].isna(), "Store"].nunique())
for c in ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]:
    df[c] = df[c].fillna(0).astype("int64")

# 5. Promo2 : vide exactement quand Promo2 = 0 -> 0 / "Aucun"
cols = ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]
incoh = ((df["Promo2"] == 0) & df[cols].notna().any(axis=1)).sum() + ((df["Promo2"] == 1) & df[cols].isna().any(axis=1)).sum()
print("Promo2 : lignes à 0 =", (df["Promo2"] == 0).sum(), "| incohérences :", incoh)
for c in ["Promo2SinceWeek", "Promo2SinceYear"]:
    df[c] = df[c].fillna(0).astype("int64")
df["PromoInterval"] = df["PromoInterval"].fillna("Aucun")

# 6. Types finaux
for c in ["StateHoliday", "PromoInterval"]:
    df[c] = df[c].astype("category")
for c in ["Store", "DayOfWeek", "Promo", "SchoolHoliday", "Promo2"]:
    df[c] = df[c].astype("int64")

assert df.isna().sum().sum() == 0 and not df.duplicated().any()
print("\nFinal :", df.shape, "| manquants : 0 | doublons : 0")
df.dtypes

Sales manquantes supprimées : 804 -> 803252 lignes
StateHoliday manquants : 7237 dont 'c' : 11
CompetitionDistance : 52767 manquants, 14594 récupérés via le magasin, 38173 remplis par la médiane (37583.38) pour 53 magasins
CompetitionOpenSince* manquants : 117412 | magasins : 163
Promo2 : lignes à 0 = 505696 | incohérences : 0

Final : (803252, 16) | manquants : 0 | doublons : 0


Store                                 int64
Date                         datetime64[us]
DayOfWeek                             int64
Sales                                 int64
Promo                                 int64
StateHoliday                       category
SchoolHoliday                         int64
StoreType                          category
Assortment                         category
CompetitionDistance                 float64
CompetitionOpenSinceMonth             int64
CompetitionOpenSinceYear              int64
Promo2                                int64
Promo2SinceWeek                       int64
Promo2SinceYear                       int64
PromoInterval                      category
dtype: object